# 02 — Equities Velocity Scan

Full S&P sector + mega cap velocity analysis with charts.
No API key needed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'specvel'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
sns.set_theme(style='darkgrid')

from adapters.equities import EquitiesAdapter
from core import compute_velocity
from leaderboard import run_leaderboard, print_leaderboard, save_leaderboard

START = '2022-01-01'
END   = '2026-03-10'

In [ ]:
adapter  = EquitiesAdapter()
df       = run_leaderboard(adapter, start=START, end=END,
                           asset_class='equities', top_n=30)
print_leaderboard(df, title='EQUITIES VELOCITY LEADERBOARD')

## Conviction Heatmap — Sectors

In [ ]:
sector_tickers = ['XLK','XLF','XLE','XLV','XLI','XLU','XLB','XLP','XLRE','XLY','XLC']
df_sectors = df[df['series_id'].isin(sector_tickers)].copy()

fig, ax = plt.subplots(figsize=(12, 4))
heat_data = df_sectors.set_index('label')[['conviction']].T
cmap = sns.diverging_palette(10, 133, as_cmap=True)
sns.heatmap(heat_data, annot=True, fmt='+.0f', cmap=cmap,
            center=0, vmin=-4, vmax=4, linewidths=0.5,
            cbar_kws={'label': 'Conviction Score'}, ax=ax)
ax.set_title('Sector Velocity Conviction Scores', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/equities_sector_heatmap.png', dpi=150)
plt.show()

## Velocity Chart — Top Movers

In [ ]:
# Plot velocity for top 4 highest conviction assets
top4 = df.head(4)['series_id'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, ticker in enumerate(top4):
    try:
        raw    = adapter.fetch(ticker, START, END)
        normed = adapter.normalize(raw)
        vel    = compute_velocity(normed)
        label  = adapter.label(ticker)
        row    = df[df['series_id'] == ticker].iloc[0]
        conv   = row['conviction']
        signal = row['signal']

        ax = axes[i]
        ax2 = ax.twinx()

        ax.plot(normed.index, normed.values, color='steelblue', alpha=0.6,
                linewidth=1.5, label='Price (normalized)')
        color = 'green' if conv > 0 else ('red' if conv < 0 else 'gray')
        ax2.plot(vel.index, vel.values, color=color, linewidth=1.2,
                 label='Velocity')
        ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
        ax2.fill_between(vel.index, vel.values, 0,
                         where=(vel.values > 0), alpha=0.15, color='green')
        ax2.fill_between(vel.index, vel.values, 0,
                         where=(vel.values < 0), alpha=0.15, color='red')

        ax.set_title(f'{label}\n{signal}  (conviction {conv:+d})',
                     fontsize=10, fontweight='bold')
        ax.set_ylabel('Price (norm)', color='steelblue', fontsize=8)
        ax2.set_ylabel('Velocity', color=color, fontsize=8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)
    except Exception as e:
        axes[i].set_title(f'{ticker}: {e}')

plt.suptitle('Top Movers — Price vs Spectral Velocity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/equities_top_movers.png', dpi=150)
plt.show()

In [ ]:
save_leaderboard(df, '../results/equities_leaderboard.csv')
print('Saved.')